In [ ]:
(* load the setup of the notebooks *)
#use "../setup.ml";;

# MiniSat

[MiniSat](http://minisat.se) is a SAT solver written in C++ by Niklas Eén and Niklas Sörensson.  The TypeScript version of this notebook uses the npm package [logic-solver](https://github.com/meteor/logic-solver), which is MiniSat compiled to JavaScript.  For OCaml, the opam package [minisat](https://github.com/c-cube/ocaml-minisat) by Simon Cruanes provides *bindings* to MiniSat, i.e. OCaml functions that call the C++ code of MiniSat.  The package contains MiniSat itself, so nothing else has to be installed.  It can be installed via
```
opam install minisat
```
In the notebook, the library is loaded with the directive `#require`.

In [ ]:
#require "minisat";;

The module `Minisat` works on the level of *clauses*:

- A *solver* is created via `Minisat.create ()`.  A solver has a state that is changed by the functions below.
- Propositional variables are represented by positive integers.  `Minisat.Lit.make n` returns the literal of the variable `n`, and `Minisat.Lit.neg l` returns the negation of the literal `l`.
- `Minisat.add_clause_l solver lits` adds the clause consisting of the literals in the list `lits` to the solver.  If the solver can already see that the clauses are unsatisfiable, it raises the exception `Minisat.Unsat`.
- `Minisat.solve solver` checks whether the clauses that have been added are satisfiable.  If they are not, the exception `Minisat.Unsat` is raised.
- After a successful call of `Minisat.solve`, the function `Minisat.value solver l` returns the value of the literal `l` in the solution that has been found.  This value is either `Minisat.V_true`, `Minisat.V_false`, or `Minisat.V_undef`.

In contrast to logic-solver, the module `Minisat` neither supports variable names nor formulas that are not clauses.  Therefore, we write a few functions that allow us to use variable names.

A literal is either a propositional variable or a negated propositional variable.  As in the notebook `04-CNF.ipynb`, the type `literal` represents literals.  In logic-solver, the negation of the variable `q` is written as `"-q"`.

In [ ]:
type literal =
  | Pos of string    (* p *)
  | Neg of string    (* ¬p *)

The type `solver` combines a MiniSat solver with a table that maps variable names to the numbers used by MiniSat:
- `sat` is the MiniSat solver,
- `numbers` is a *hash table* that maps every variable name that has been used so far to its number.  A hash table of type `(string, int) Hashtbl.t` is a mutable map from strings to integers.
- `consistent` is set to `false` as soon as MiniSat has detected that the clauses are unsatisfiable while they are added.  The keyword `mutable` allows this field to be changed.

In [ ]:
type solver = {
  sat                : Minisat.t;
  numbers            : (string, int) Hashtbl.t;
  mutable consistent : bool;
}

The function `new_solver` takes the argument `()`.  It returns a new solver that does not contain any clauses.

In [ ]:
let new_solver () : solver =
  { sat = Minisat.create (); numbers = Hashtbl.create 1024; consistent = true }

The function `variable_number` takes two arguments:
- `s` is a solver,
- `p` is the name of a propositional variable.

It returns the number of `p`.  If `p` has not been used before, it is assigned the next free number.  As the numbers start at 1, this is the number of variables used so far plus 1.

In [ ]:
let variable_number (s : solver) (p : string) : int =
  match Hashtbl.find_opt s.numbers p with
  | Some n -> n
  | None   ->
      let n = Hashtbl.length s.numbers + 1 in
      Hashtbl.add s.numbers p n;
      n

The function `to_lit` takes two arguments:
- `s` is a solver,
- `l` is a literal.

It returns the MiniSat literal corresponding to `l`.

In [ ]:
let to_lit (s : solver) (l : literal) : Minisat.Lit.t =
  match l with
  | Pos p -> Minisat.Lit.make (variable_number s p)
  | Neg p -> Minisat.Lit.neg (Minisat.Lit.make (variable_number s p))

The function `require` takes two arguments:
- `s` is a solver,
- `clauses` is a list of clauses, where every clause is a list of literals.

It adds the clauses to the solver.  This corresponds to the method `require` of logic-solver.  If MiniSat raises the exception `Minisat.Unsat`, the field `consistent` is set to `false`.

In [ ]:
let require (s : solver) (clauses : literal list list) : unit =
  List.iter
    (fun clause ->
       try Minisat.add_clause_l s.sat (List.map (to_lit s) clause)
       with Minisat.Unsat -> s.consistent <- false)
    clauses

The function `solve` takes one argument:
- `s` is a solver.

It checks whether the clauses of `s` are satisfiable.  The result has the type `string list option`:
- If the clauses are satisfiable, `Some vars` is returned, where `vars` is the sorted list of those variables that are true in the solution found by MiniSat.  This corresponds to the method `getTrueVars` of logic-solver.
- Otherwise, `None` is returned.

**Implementation:** `Hashtbl.fold f table init` calls `f name number acc` for every entry of the hash table.  Here, it collects the names of the variables that are true.  `List.sort compare` sorts these names.

In [ ]:
let solve (s : solver) : string list option =
  if not s.consistent then
    None
  else
    try
      Minisat.solve s.sat;
      let true_vars =
        Hashtbl.fold
          (fun name n acc ->
             if Minisat.value s.sat (Minisat.Lit.make n) = Minisat.V_true then name :: acc else acc)
          s.numbers []
      in
      Some (List.sort compare true_vars)
    with Minisat.Unsat -> None

As an example, the formula
$$ (\neg p \vee  q \vee \neg r) \wedge (p \vee \neg q \vee r) \wedge (p \vee q \vee \neg r) \wedge (p \vee q \vee r), $$
which is in conjunctive normal form, is represented by the following code:

In [ ]:
let solver = new_solver ();;
require solver [[Neg "p"; Pos "q"; Neg "r"];
                [Pos "p"; Neg "q"; Pos "r"];
                [Pos "p"; Pos "q"; Neg "r"];
                [Pos "p"; Pos "q"; Pos "r"]]

In [ ]:
let sol = solve solver

This shows that the formula `f` is satisfiable.  The variables in the list are true, all other variables are false.  Hence, the propositional valuation
$$ \mathcal{I} = \{ p \mapsto \texttt{True}, q \mapsto \texttt{True}, r \mapsto \texttt{True} \} $$
is a solution of `f`, i.e. we have
$$ \mathcal{I}(\texttt{f}) = \texttt{True}. $$
The formula has more than one solution.  Therefore, logic-solver, which is based on a different version of MiniSat, finds a different solution.

## Sudoku

The function `range` takes two arguments:
- `start` is an integer,
- `stop` is an integer.

It returns the list $[ \texttt{start}, \cdots, \texttt{stop} ]$.

In [ ]:
let range (start : int) (stop : int) : int list =
  List.init (stop - start + 1) (fun i -> start + i)

The Finnish mathematician Arto Inkala claims to have created the [hardest sudoku](https://abcnews.go.com/blogs/headlines/2012/06/can-you-solve-the-hardest-ever-sudoku) ever.  It is shown below.

<img src="sudoku.png" alt="A game of Sudoku" width="50%">

The function `create_puzzle` takes the argument `()`.  It returns the Sudoku as a list of rows, where every row is a list of 9 cells.  A cell that contains the digit `d` is represented as `Some d`, a cell that is left unspecified is represented as `None`.  In order to make the puzzle easy to read, every row is written as a string that contains a `'*'` for every unspecified cell.

In [ ]:
let create_puzzle () : int option list list =
  let rows = ["8********";
              "**36*****";
              "*7**9*2**";
              "*5***7***";
              "****457**";
              "***1***3*";
              "**1****68";
              "**85***1*";
              "*9****4**"] in
  let cell (ch : char) : int option =
    if ch = '*' then None else Some (Char.code ch - Char.code '0')
  in
  List.map (fun row -> List.map cell (List.of_seq (String.to_seq row))) rows

We use the following variables:
* `Q<r,c,d>` is a Boolean variable stating that the field in row `r` and column `c` holds the digit `d`.
  Here, `r`, `c`, `d` are all elements from the set $\{1,\cdots,9\}$.

The function `var_name` takes three arguments:
- `row` is the number of a row,
- `col` is the number of a column,
- `digit` is a digit.

It returns the string `Q<row,col,digit>`, which is interpreted as a variable name.

In [ ]:
let var_name (row : int) (col : int) (digit : int) : string =
  Printf.sprintf "Q<%d,%d,%d>" row col digit

In [ ]:
var_name 1 2 3

The function `complement` takes one argument:
- `l` is a literal.

It computes the complement of `l`.  If $p$ is a propositional variable, we have the following:
* $\texttt{complement}(p) = \neg p$,
* $\texttt{complement}(\neg p) = p$.

In [ ]:
let complement (l : literal) : literal =
  match l with
  | Pos p -> Neg p
  | Neg p -> Pos p

logic-solver provides the function `Logic.exactlyOne`, which creates a formula stating that exactly one of the given variables is true.  The module `Minisat` has no such function.  Therefore, we define it ourselves.  The functions below represent sets of clauses as lists of lists of literals.

The function `at_most_one` takes one argument:
- `vars` is a list of propositional variables.

It returns a list of clauses stating that at most one of the variables in `vars` is true.  For every pair of variables $p$ and $q$ where $p$ occurs before $q$ in the list, the clause $\{\neg p, \neg q\}$ is created.

**Implementation:** The recursive function `pairs` takes the remaining list of variables.  If this list has the form `p :: rest`, it creates the clauses for `p` and all variables `q` in `rest` and continues with `rest`.

In [ ]:
let at_most_one (vars : string list) : literal list list =
  let rec pairs = function
    | []        -> []
    | p :: rest -> List.map (fun q -> [Neg p; Neg q]) rest @ pairs rest
  in
  pairs vars

In [ ]:
at_most_one ["a"; "b"; "c"]

The function `exactly_one` takes one argument:
- `vars` is a list of propositional variables.

It returns a list of clauses stating that exactly one of the variables in `vars` is true: At least one of the variables is true, and at most one is true.

In [ ]:
let exactly_one (vars : string list) : literal list list =
  List.map (fun p -> Pos p) vars :: at_most_one vars

In [ ]:
exactly_one ["a"; "b"; "c"]

A `position` is a pair of the form `(row, col)`, where `row` is the row
of a cell and `col` is its column.

In [ ]:
type position = int * int

The function `exactly_one_for_digit` takes two arguments:
* `l` is a list of positions of a Sudoku.  These are pairs of the form `(row, col)`, where both `row` and `col` are elements of the set $\{1, \cdots, 9\}$.
* `digit` is an element from the set $\{1,\cdots,9\}$.

It returns a list of clauses expressing the fact that the given `digit`
appears exactly once among the fields specified by the positions in `l`.

In [ ]:
let exactly_one_for_digit (l : position list) (digit : int) : literal list list =
  exactly_one (List.map (fun (row, col) -> var_name row col digit) l)

The function `exactly_one_for_every_digit` takes one argument:
- `l` is a list of positions of a Sudoku.

It returns a list of clauses expressing that all digits from the set
$\{1,\cdots,9\}$ occur exactly once in the list `l` of Sudoku cells.  `List.concat_map f l` applies `f` to every element of `l` and concatenates the resulting lists.

In [ ]:
let exactly_one_for_every_digit (l : position list) : literal list list =
  List.concat_map (exactly_one_for_digit l) (range 1 9)

The function `exactly_one_digit` takes two arguments:
- `row` is the number of a row,
- `col` is the number of a column.

These specify a field of the Sudoku.  The function returns a list of clauses specifying that exactly one of the variables

* `Q<row,col,1>`, `Q<row,col,2>`, $\cdots$, `Q<row,col,9>`

is true.

In [ ]:
let exactly_one_digit (row : int) (col : int) : literal list list =
  exactly_one (List.map (var_name row col) (range 1 9))

The function `constraints_from_puzzle` takes the argument `()`.  It returns a list of unit clauses stating that the variables corresponding to numbers that are already given in the Sudoku puzzle take the values that are specified.

**Implementation:** `List.concat (List.mapi f l)` calls `f i x` for every element `x` of `l` at position `i` and concatenates the results.  For every cell `Some d` in row `r` and column `c`, the unit clause $\{\texttt{Q<}r\texttt{,}c\texttt{,}d\texttt{>}\}$ is created.  As the positions start at 0, they are increased by 1.

In [ ]:
let constraints_from_puzzle () : literal list list =
  List.concat
    (List.mapi
       (fun r row ->
          List.concat
            (List.mapi
               (fun c cell ->
                  match cell with
                  | Some d -> [[Pos (var_name (r + 1) (c + 1) d)]]
                  | None   -> [])
               row))
       (create_puzzle ()))

The function `get_block_cells` calculates the absolute grid coordinates for all 9 cells
within a specific 3x3 Sudoku block.  The Sudoku board is divided into a 3x3 grid of blocks.
This function takes the zero-based indices of a block and maps its local cells to the
1-based coordinates used by the solver variables.  It takes two arguments:
 * `r` is the row index of the 3x3 block (0, 1, or 2),
 * `c` is the column index of the 3x3 block (0, 1, or 2).

The function returns the list of the 9 positions `(row, col)` of the cells of this block on the full 9x9 Sudoku board.

For example, the coordinates of the top-middle block (`r = 0, c = 1`) are given as follows:
```
get_block_cells 0 1 = [(1, 4); (1, 5); (1, 6);
                       (2, 4); (2, 5); (2, 6);
                       (3, 4); (3, 5); (3, 6)]
```

In [ ]:
let get_block_cells (r : int) (c : int) : position list =
  List.concat_map (fun row -> List.map (fun col -> (r * 3 + row, c * 3 + col)) (range 1 3)) (range 1 3)

In [ ]:
get_block_cells 0 1

The function `all_constraints` takes one argument:
- `solver` is a solver.

It adds the clauses that encode the given Sudoku to `solver`, solves them, and returns the result of `solve`.

In [ ]:
let all_constraints (solver : solver) : string list option =
  (* 1. Start with constraints from the puzzle *)
  require solver (constraints_from_puzzle ());
  (* 2. There is exactly one digit in every field *)
  List.iter (fun row -> List.iter (fun col -> require solver (exactly_one_digit row col)) (range 1 9))
            (range 1 9);
  (* 3. All entries in a row are unique *)
  List.iter (fun row -> require solver (exactly_one_for_every_digit (List.map (fun col -> (row, col)) (range 1 9))))
            (range 1 9);
  (* 4. All entries in a column are unique *)
  List.iter (fun col -> require solver (exactly_one_for_every_digit (List.map (fun row -> (row, col)) (range 1 9))))
            (range 1 9);
  (* 5. All entries in a 3x3 square are unique *)
  List.iter (fun r -> List.iter (fun c -> require solver (exactly_one_for_every_digit (get_block_cells r c))) (range 0 2))
            (range 0 2);
  solve solver

In [ ]:
let sudoku_solver = new_solver ();;
let solution = time "solve" (fun () -> all_constraints sudoku_solver)

This Sudoku is modelled using $9^3 = 729$ propositional variables.  MiniSat solves the problem in a few milliseconds.  The function `Minisat.n_clauses` returns the number of clauses that the solver still keeps.  As MiniSat simplifies the clauses while they are added, this number is smaller than the number of clauses that we have added.

In [ ]:
(Hashtbl.length sudoku_solver.numbers, Minisat.n_clauses sudoku_solver.sat)

## Graphical Representation

In order to display HTML in the notebook, we load the library `jupyter.notebook`, which provides the function `Jupyter_notebook.display`.

In [ ]:
#require "jupyter.notebook";;

The function `show_solution` takes two arguments:
- `solution` is the list of variables that are true in the solution,
- `width` is the width of the table as a percentage of the width of the notebook.  It is an optional argument with the default value `"50%"`.

It displays the solution as an HTML table.  The digits that are given in the puzzle are shown in bold face.

**Implementation:** `Scanf.sscanf s format f` reads the values described by `format` from the string `s` and passes them to the function `f`.  Here, the format `"Q<%d,%d,%d>"` extracts the row, the column, and the digit from a variable name.  The resulting association list maps positions to digits.

In [ ]:
let show_solution ?(width = "50%") (solution : string list) : unit =
  let puzzle = create_puzzle () in
  let solution_map =
    List.map (fun var -> Scanf.sscanf var "Q<%d,%d,%d>" (fun row col digit -> ((row, col), digit))) solution
  in
  let html = Buffer.create 8192 in
  Buffer.add_string html
    (Printf.sprintf "<table style=\"width:%s; border-collapse: collapse; border: 2px solid black; font-family: sans-serif;\">" width);
  for row = 0 to 8 do
    Buffer.add_string html "<tr>";
    for col = 0 to 8 do
      let (value, cell_style) =
        match List.nth (List.nth puzzle row) col with
        | Some d -> (Some d, "font-weight: bold;")
        | None   -> (List.assoc_opt (row + 1, col + 1) solution_map, "font-weight: normal;")
      in
      let is_gray  = (row / 3 + col / 3) mod 2 <> 0 in
      let bg_color = if is_gray then "#f0f0f0" else "#ffffff" in
      let border_style =
        "border: 1px solid #ccc;"
        ^ (if (col + 1) mod 3 = 0 && col < 8 then "border-right: 2px solid black;" else "")
        ^ (if (row + 1) mod 3 = 0 && row < 8 then "border-bottom: 2px solid black;" else "")
      in
      let text = match value with Some d -> string_of_int d | None -> "" in
      Buffer.add_string html
        (Printf.sprintf "<td style=\"%s width:30px; height:30px; text-align:center; font-size:20px; background-color:%s; %s\">%s</td>"
           border_style bg_color cell_style text)
    done;
    Buffer.add_string html "</tr>"
  done;
  Buffer.add_string html "</table>";
  ignore (Jupyter_notebook.display "text/html" (Buffer.contents html))

In [ ]:
match solution with
| Some vars -> show_solution vars
| None      -> print_endline "The Sudoku is not solvable."